# CLM fine-tuning on Kaggle

Fine-tunes all-MiniLM-L6-v2 on the contemplative-neuroscience corpus and compares it with the base model on held-out papers.

**Setup (right-hand panel → Session options):**
- Accelerator: **GPU T4 x2** (or P100)
- Internet: **On** (needed to download the base model; requires a phone-verified Kaggle account)
- Input: add the uploaded dataset `clm-kaggle-bundle` (the zip with `clm/` and `clm/data/corpus.jsonl`)

Then **Run All**. When it finishes, download `clm-output.zip` from the Output panel.

In [ ]:
# Copy the bundle somewhere writable (/kaggle/input is read-only)
import shutil, subprocess, os
from pathlib import Path

import zipfile
REPO = Path('/kaggle/working/repo')
if REPO.exists(): shutil.rmtree(REPO)
hits = list(Path('/kaggle/input').rglob('clm/common.py'))
if hits:                                   # Kaggle unpacked the zip
    src = hits[0].parents[1]
    shutil.copytree(src, REPO)
else:                                      # zip kept as a file
    zips = list(Path('/kaggle/input').rglob('clm-kaggle-bundle*.zip'))
    assert zips, 'Bundle not found: add the clm-kaggle-bundle dataset as input'
    src = zips[0]
    zipfile.ZipFile(src).extractall(REPO)
os.chdir(REPO)
os.environ['HF_HUB_DISABLE_XET'] = '1'   # avoid hanging Xet downloads
print('bundle from', src)
print('corpus lines:', sum(1 for _ in open('clm/data/corpus.jsonl', encoding='utf-8')))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, sentence_transformers
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| sentence-transformers', sentence_transformers.__version__)
assert torch.cuda.is_available(), 'No GPU: set Accelerator to GPU in Session options'

In [ ]:
# Fine-tune (defaults: 3 epochs, batch 64, lr 2e-5). Writes clm/models/clm-neuro-minilm and clm/data/test_ids.json
!python clm/train/finetune.py

In [ ]:
# Base vs fine-tuned vs SPECTER2 on held-out papers. Writes clm/eval/RESULTS.md and results.json
!python clm/eval/evaluate.py --models sentence-transformers/all-MiniLM-L6-v2 clm/models/clm-neuro-minilm allenai/specter2_base
print(open('clm/eval/RESULTS.md').read())

In [ ]:
# Package everything needed back on the local machine
import zipfile
out = Path('/kaggle/working/clm-output.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in Path('clm/models/clm-neuro-minilm').rglob('*'):
        if p.is_file(): z.write(p, p.as_posix())
    for p in ['clm/data/test_ids.json', 'clm/eval/RESULTS.md', 'clm/eval/results.json']:
        z.write(p, p)
os.chdir('/kaggle/working')
shutil.rmtree(REPO)  # corpus has non-redistributable abstracts; keep only the zip in Output
print(out, round(out.stat().st_size / 1e6, 1), 'MB')